# OpenCode — architecture & live demo

Core: `packages/opencode/src` (TypeScript, **Effect** runtime). Condensed harness:
`mini-agent/miniagent` (Python). Full write-up: `../01-overview-and-thesis.md`,
`../06-code-walkthrough-and-callgraphs.md`.

## Architecture in one screen

- **Two cores:** V1 `SessionPrompt.runLoop` (`session/prompt.ts:1081`, a literal `while(true)`);
  V2 durable `SessionRunner.run` (`packages/core/src/session/runner/llm.ts:383`).

- **One provider turn:** `SessionProcessor.process` (`session/processor.ts:627`) drains a normalized
  `LLMEvent` stream (Vercel AI SDK `streamText`).

- **Message-of-typed-parts** with a tool state machine `pending→running→completed|error`
  (`schema/session-message.ts`).

- **Permission engine** allow/ask/deny, wildcard, last-match-wins, default *ask* (`permission/index.ts`).

- **Per-turn shadow-git snapshots + revert** (`snapshot/index.ts`).

- **SQLite/Drizzle** transcript is the source of truth.

- Sub-agents via the **`task` tool** (child session, derived permissions); per-model prompt `.txt` files.

In [ ]:
import os, sys

def find_repo_root():
    markers = {'mini-agent', 'pi', 'openhands_all'}
    d = os.path.abspath(os.getcwd())
    while True:
        try:
            if markers.issubset(set(os.listdir(d))):
                return d
        except OSError:
            pass
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    for cand in ('/repo', os.path.expanduser('~')):
        if os.path.isdir(os.path.join(cand, 'mini-agent')):
            return cand
    raise RuntimeError('repo root not found; set REPO manually')

REPO = find_repo_root()
print('REPO =', REPO)

In [ ]:
# Helper: print a slice of a REAL core source file around a symbol, so every
# architectural claim in this notebook can be checked against the implementation.
def show(relpath, needle=None, ctx=6, max_lines=40):
    path = os.path.join(REPO, relpath)
    with open(path, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    print(f'# {relpath}  ({len(lines)} lines)')
    if needle is None:
        start, end = 0, min(max_lines, len(lines))
    else:
        idx = next((i for i, l in enumerate(lines) if needle in l), None)
        if idx is None:
            print(f'  (needle {needle!r} not found)')
            return
        start, end = max(0, idx - 1), min(len(lines), idx + ctx)
    for i in range(start, end):
        print(f'{i+1:5} | {lines[i].rstrip()}')

### Verify against the real core

The claims above are checkable. Print the actual loop, the tool state machine, and the
permission engine from the shipping TypeScript source:

In [ ]:
show('packages/opencode/src/session/prompt.ts', 'while (true)', ctx=10)

In [ ]:
show('packages/opencode/src/permission/index.ts', 'export function evaluate', ctx=12)

In [ ]:
# Snapshot (per-turn git) + task (sub-agent) tools exist as real files:
for rel in ['packages/opencode/src/snapshot/index.ts',
            'packages/opencode/src/tool/task.ts',
            'packages/opencode/src/tool/registry.ts']:
    p = os.path.join(REPO, rel)
    print(f'{rel:52} {"exists" if os.path.exists(p) else "MISSING"}')

## Live demo (condensed harness, offline)

The `mini-agent` distillation runs the *same turn shape* with a scripted provider — no
network, no API key. Watch: user → assistant(tool_call `write`) → tool result → assistant(final),
with permission allow and a full event trace.

In [ ]:
import asyncio, tempfile
sys.path.insert(0, os.path.join(REPO, 'mini-agent'))
from miniagent import LocalSession, Rule, ScriptedProvider
from miniagent.types import Finish, TextDelta, ToolCallRequest

def run_async(coro):
    try:
        running = asyncio.get_running_loop()
    except RuntimeError:
        running = None
    if running and running.is_running():
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor(1) as ex:
            return ex.submit(lambda: asyncio.run(coro)).result()
    return asyncio.run(coro)

provider = ScriptedProvider(turns=[
    [TextDelta(text='Creating the file.'),
     ToolCallRequest(call_id='c1', tool='write', args={'path': 'hello.txt', 'content': 'hi\n'}),
     Finish(reason='tool_calls')],
    [TextDelta(text='Done, hello.txt created.'), Finish(reason='stop')],
])
workdir = tempfile.mkdtemp(prefix='opencode_demo_')
session = LocalSession(provider=provider, workdir=workdir,
                       rules=[Rule(action='write', pattern='*', decision='allow')])
final = run_async(session.prompt('create hello.txt'))
print('FINAL:', final.text())
print('\nTRANSCRIPT:')
for m in session.store.history(session.id):
    tc = [p.tool for p in m.tool_calls()] if hasattr(m, 'tool_calls') else []
    print(f'  {m.role:10} tools={tc} {m.text()[:50]!r}')
print('\nEVENT TRACE:')
for e in session.trace.read(session.id):
    print('  ', e.type, {k: v for k, v in e.data.items() if k != 'session_id'})

**What you just saw** maps 1:1 to the real core: one model turn per iteration, tool result
re-enters via the store, permission gate on the `write` tool, failures/denials become data,
and every step publishes an event. The real OpenCode adds SQLite, snapshots, LSP, and the
Effect runtime on top of this exact shape.